In [1]:
%pip install rarfile

Note: you may need to restart the kernel to use updated packages.


In [8]:
from pathlib import Path
import shutil
import sys

SOURCE_DIR = Path(r"C:\Users\75448\Downloads\知识库历史文件")
OUTPUT_DIR = Path(r"C:\Users\75448\Downloads\output_dir")

SEVEN_ZIP = next((str(p) for p in [
    Path(r"C:\Program Files\7-Zip\7z.exe"),
    Path(r"C:\Program Files (x86)\7-Zip\7z.exe"),
] if p.is_file()), None)
SEVEN_ZIP = SEVEN_ZIP or shutil.which('7z') or shutil.which('7zz') or shutil.which('7za')

print('Python:', sys.version.split()[0])
print('原文件夹:', SOURCE_DIR)
print('新文件夹:', OUTPUT_DIR)
print('7-Zip:', SEVEN_ZIP or '未发现（RAR 无法解压）')
assert sys.version_info >= (3, 11), '需要 Python 3.11 或更新版本'
assert SOURCE_DIR.is_dir(), 'SOURCE_DIR 文件夹不存在，请检查路径'
assert SOURCE_DIR.resolve() != OUTPUT_DIR.resolve(), '输入输出不能相同'


Python: 3.12.13
原文件夹: C:\Users\75448\Downloads\知识库历史文件
新文件夹: C:\Users\75448\Downloads\output_dir
7-Zip: C:\Program Files\7-Zip\7z.exe


In [9]:
import subprocess
if SEVEN_ZIP:
    check = subprocess.run([str(SEVEN_ZIP)], capture_output=True, text=True, errors='replace', timeout=30)
    print('找到 7-Zip，可调用命令行。')
    print((check.stdout or check.stderr).splitlines()[:5])
else:
    print('未找到 7z 命令：ZIP 仍可解压，RAR 会记录失败。')

找到 7-Zip，可调用命令行。
['', '7-Zip 26.04 (x64) : Copyright (c) 1999-2026 Igor Pavlov : 2026-10-05', '', 'Usage: 7z <command> [<switches>...] <archive_name> [<file_names>...] [@listfile]', '']


In [10]:
from pathlib import Path
import csv
import re
import shutil
import stat
import subprocess
import tempfile
import zipfile

# 运行本代码前，Notebook 已定义 SOURCE_DIR、OUTPUT_DIR、SEVEN_ZIP。
ARCHIVE_SUFFIXES = {'.zip', '.rar'}
MAX_DEPTH = 10
MAX_FILE_BYTES = 5 * 1024**3         # 每个文件最多 5 GiB，可按需要调整
MAX_TOTAL_BYTES = 200 * 1024**3      # 累计输出最多 200 GiB，可按需要调整
MAX_ZIP_RATIO = 2000                 # 检测异常高压缩比
MAX_MEMBERS_PER_ZIP = 50000
RAR_TIMEOUT_SECONDS = 3600

manifest = []
errors = []
archive_count = 0
total_bytes = 0


def record_error(origin, detail):
    errors.append({'source': str(origin), 'error': str(detail)})


def clean_filename(name):
    """只保留文件名，不创建包内子目录；兼容 Windows 非法文件名。"""
    name = str(name).replace('\\', '/').split('/')[-1]
    name = re.sub(r'[<>:"/\\|?*\x00-\x1f]', '_', name).rstrip(' .')
    if not name or name in {'.', '..'}:
        return None
    if name.split('.')[0].upper() in {
        'CON', 'PRN', 'AUX', 'NUL',
        *(f'COM{i}' for i in range(1, 10)),
        *(f'LPT{i}' for i in range(1, 10))
    }:
        name = '_' + name
    if len(name) > 180:
        p = Path(name)
        name = p.stem[:160] + p.suffix[:20]
    return name


def choose_target(name):
    name = clean_filename(name)
    if not name:
        raise ValueError('无效文件名')
    result = OUTPUT_DIR / name
    n = 2
    while result.exists():
        result = OUTPUT_DIR / f'{Path(name).stem}_{n}{Path(name).suffix}'
        n += 1
    return result


def copy_bounded(source, destination, limit):
    copied = 0
    with destination.open('xb') as out:
        while True:
            chunk = source.read(1024 * 1024)
            if not chunk:
                break
            copied += len(chunk)
            if copied > limit:
                raise ValueError(f'单文件超过限制 {limit} bytes')
            out.write(chunk)
    return copied


def save_file(stream, name, origin, expected_size=None):
    global total_bytes
    if expected_size is not None and expected_size > MAX_FILE_BYTES:
        record_error(origin, f'跳过：文件过大 {expected_size} bytes')
        return
    if total_bytes >= MAX_TOTAL_BYTES:
        record_error(origin, '达到累计输出大小上限')
        return
    try:
        dest = choose_target(name)
    except Exception as exc:
        record_error(origin, exc)
        return
    try:
        available = min(MAX_FILE_BYTES, MAX_TOTAL_BYTES - total_bytes)
        copied = copy_bounded(stream, dest, available)
        if expected_size is not None and copied != expected_size:
            raise IOError(f'文件大小不一致：预计 {expected_size}，实际 {copied}')
        total_bytes += copied
        manifest.append({'source': str(origin), 'filename': dest.name, 'bytes': copied})
    except Exception as exc:
        dest.unlink(missing_ok=True)  # 防止半成品残留
        record_error(origin, exc)


def process_zip(path, origin, depth):
    try:
        # GBK 处理未标记 UTF-8 的旧 ZIP 中文文件名；UTF-8 标记仍优先。
        with zipfile.ZipFile(path, 'r', metadata_encoding='gbk') as zf:
            infos = zf.infolist()
            if len(infos) > MAX_MEMBERS_PER_ZIP:
                record_error(origin, f'成员数量超过 {MAX_MEMBERS_PER_ZIP}')
                return
            for info in infos:
                if info.is_dir():
                    continue
                if stat.S_ISLNK(info.external_attr >> 16):
                    record_error(f'{origin} -> {info.filename}', '跳过符号链接')
                    continue
                member_origin = f'{origin} -> {info.filename}'
                if info.flag_bits & 0x1:
                    record_error(member_origin, '加密 ZIP 成员：需要密码')
                    continue
                if info.file_size > MAX_FILE_BYTES:
                    record_error(member_origin, '跳过超大文件')
                    continue
                if (info.file_size > 0 and info.file_size / max(info.compress_size, 1)
                        > MAX_ZIP_RATIO):
                    record_error(member_origin, '压缩比过高，已跳过')
                    continue
                extension = Path(info.filename.replace('\\', '/')).suffix.lower()
                try:
                    with zf.open(info) as stream:
                        if extension in ARCHIVE_SUFFIXES:
                            with tempfile.TemporaryDirectory(prefix='nested_zip_') as tmp:
                                nested = Path(tmp) / ('nested' + extension)
                                with nested.open('wb') as out:
                                    shutil.copyfileobj(stream, out, 1024 * 1024)
                                process_archive(nested, member_origin, depth + 1)
                        else:
                            save_file(stream, info.filename, member_origin, info.file_size)
                except Exception as exc:
                    record_error(member_origin, f'读取 ZIP 成员失败：{exc}')
    except (UnicodeDecodeError, zipfile.BadZipFile) as exc:
        record_error(origin, f'ZIP 读取失败（可能有编码或压缩包损坏问题）：{exc}')
    except Exception as exc:
        record_error(origin, f'ZIP 解压异常：{exc}')


def process_rar(path, origin, depth):
    if not SEVEN_ZIP:
        record_error(origin, '未找到 7z.exe。请安装 7-Zip 并配置 SEVEN_ZIP')
        return
    try:
        # 7z e 扁平解压，-aou 避免包内重名文件相互覆盖。
        with tempfile.TemporaryDirectory(prefix='rar_flat_') as tmp:
            args = [str(SEVEN_ZIP), 'e', str(path), f'-o{tmp}', '-aou', '-y', '-bd', '-bb0']
            try:
                done = subprocess.run(
                    args, stdin=subprocess.DEVNULL, capture_output=True,
                    text=True, errors='replace', timeout=RAR_TIMEOUT_SECONDS
                )
            except subprocess.TimeoutExpired:
                record_error(origin, f'RAR 解压超时：{RAR_TIMEOUT_SECONDS} 秒')
                return
            if done.returncode != 0:
                detail = (done.stderr + '\n' + done.stdout).strip()[-1500:]
                record_error(origin, f'7-Zip 返回码 {done.returncode}：{detail}')
            extracted = sorted(Path(tmp).rglob('*'))
            for file in extracted:
                if not file.is_file() or file.is_symlink():
                    continue
                member_origin = f'{origin} -> {file.name}'
                try:
                    if file.suffix.lower() in ARCHIVE_SUFFIXES:
                        process_archive(file, member_origin, depth + 1)
                    else:
                        with file.open('rb') as stream:
                            save_file(stream, file.name, member_origin, file.stat().st_size)
                except Exception as exc:
                    record_error(member_origin, f'处理 RAR 解压结果失败：{exc}')
    except Exception as exc:
        record_error(origin, f'调用 7-Zip 失败：{exc}')


def process_archive(path, origin, depth=0):
    global archive_count
    if depth >= MAX_DEPTH:
        record_error(origin, f'压缩包嵌套超过 {MAX_DEPTH} 层')
        return
    archive_count += 1
    if path.suffix.lower() == '.zip':
        process_zip(path, origin, depth)
    elif path.suffix.lower() == '.rar':
        process_rar(path, origin, depth)


def run_flatten():
    if not SOURCE_DIR.is_dir():
        raise FileNotFoundError(f'输入目录不存在：{SOURCE_DIR}')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    if any(OUTPUT_DIR.iterdir()):
        raise RuntimeError('输出目录不为空。请使用一个新的空目录，避免重复副本或旧结果混入。')
    if SOURCE_DIR.resolve() == OUTPUT_DIR.resolve() or (
        OUTPUT_DIR.resolve().is_relative_to(SOURCE_DIR.resolve()) or
        SOURCE_DIR.resolve().is_relative_to(OUTPUT_DIR.resolve())
    ):
        raise ValueError('输入和输出文件夹不能相同或互相包含')
    for file in sorted(SOURCE_DIR.rglob('*')):
        if not file.is_file() or file.is_symlink():
            continue
        origin = str(file.relative_to(SOURCE_DIR))
        if file.suffix.lower() in ARCHIVE_SUFFIXES:
            process_archive(file, origin)
        else:
            try:
                with file.open('rb') as stream:
                    save_file(stream, file.name, origin, file.stat().st_size)
            except Exception as exc:
                record_error(origin, f'复制失败：{exc}')
    # CSV 放在输出目录的旁边，确保输出目录只有待后续处理的原始文件。
    for kind, rows, columns in [
        ('file_index', manifest, ['source', 'filename', 'bytes']),
        ('errors', errors, ['source', 'error']),
    ]:
        log_path = OUTPUT_DIR.parent / f'{OUTPUT_DIR.name}_{kind}.csv'
        with log_path.open('w', encoding='utf-8-sig', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=columns)
            writer.writeheader()
            writer.writerows(rows)
        print('日志：', log_path)
    print(f'完成：压缩包 {archive_count} 个，输出 {len(manifest)} 个文件，错误/警告 {len(errors)} 条，累计 {total_bytes / 1024**2:.1f} MiB')


run_flatten()


日志： C:\Users\75448\Downloads\output_dir_file_index.csv
日志： C:\Users\75448\Downloads\output_dir_errors.csv
完成：压缩包 370 个，输出 1943 个文件，错误/警告 0 条，累计 678.4 MiB
